In [ ]:
# CÉLULA 1: Configuração e Carregamento dos Dados
# Importamos as bibliotecas necessárias, incluindo matplotlib para visualização.
import pandas as pd
import matplotlib.pyplot as plt
import os

# Caminhos hipotéticos para os dados já processados nas etapas 01 e 02
CAMINHO_CAGED = os.path.join("..", "data", "processed", "caged", "caged_rs_filtrado.csv")
CAMINHO_CLIMA = os.path.join("..", "data", "processed", "clima", "inmet_rs_filtrado.csv")

# Carregamento seguro
try:
    df_caged = pd.read_csv(CAMINHO_CAGED)
    df_clima = pd.read_csv(CAMINHO_CLIMA)
    print(f"✅ Dados carregados! CAGED: {df_caged.shape[0]} linhas | INMET: {df_clima.shape[0]} linhas")
except FileNotFoundError as e:
    print(f"❌ Erro ao carregar: {e}. Verifique se os notebooks 01 e 02 exportaram os dados corretamente.")

In [ ]:
# CÉLULA 2: Agregação Temporal (INMET)
# O CAGED informa o saldo mensal. Precisamos agrupar os dias do INMET para ter a chuva total do mês.

# Certificando que a data do clima é datetime
df_clima['Data'] = pd.to_datetime(df_clima['Data'])

# Criamos colunas de Ano e Mês para facilitar o agrupamento e o merge futuro
df_clima['Ano'] = df_clima['Data'].dt.year
df_clima['Mes'] = df_clima['Data'].dt.month

# Identifica a coluna de precipitação e soma a chuva por Ano e Mês
col_chuva = [c for c in df_clima.columns if 'PRECIPITA' in c.upper()][0]
df_clima_mensal = df_clima.groupby(['Ano', 'Mes'])[col_chuva].sum().reset_index()

# Renomeia para clareza
df_clima_mensal.rename(columns={col_chuva: 'Chuva_Total_Mes'}, inplace=True)
print("🌧️ Dados climáticos agregados por mês:")
display(df_clima_mensal.head())

In [ ]:
# CÉLULA 3: Preparação do CAGED e Junção (Merge)
# Extraímos o Ano e Mês do CAGED (ajuste o nome da coluna conforme o seu arquivo)
col_competencia = 'Competência' # Exemplo: '202312' para Dezembro de 2023

if col_competencia in df_caged.columns:
    df_caged['Ano'] = df_caged[col_competencia].astype(str).str[:4].astype(int)
    df_caged['Mes'] = df_caged[col_competencia].astype(str).str[4:6].astype(int)

# Realiza o inner join (mantém apenas os meses que existem em ambos os datasets)
df_merged = pd.merge(df_caged, df_clima_mensal, on=['Ano', 'Mes'], how='inner')

print(f"🔗 Merge concluído! Tamanho do dataset final: {df_merged.shape[0]} registros.")
display(df_merged[['Ano', 'Mes', 'Saldos', 'Chuva_Total_Mes']].head())

In [ ]:
# CÉLULA 4: Cálculo da Correlação
# Vamos verificar se existe correlação matemática entre o volume de chuva e o saldo de empregos.

col_saldo = 'Saldos' # Ajuste para a coluna exata de saldo do seu CAGED

if col_saldo in df_merged.columns and 'Chuva_Total_Mes' in df_merged.columns:
    # O método 'pearson' mede a correlação linear (varia de -1 a 1)
    correlacao = df_merged[col_saldo].corr(df_merged['Chuva_Total_Mes'], method='pearson')
    
    print(f"📈 Coeficiente de Correlação de Pearson: {correlacao:.4f}")
    if correlacao > 0.5:
        print("💡 Interpretação: Forte correlação positiva.")
    elif correlacao < -0.5:
        print("💡 Interpretação: Forte correlação negativa (mais chuva, menos emprego).")
    else:
        print("💡 Interpretação: Correlação fraca ou inexistente linearmente.")
else:
    print("❌ Colunas necessárias para correlação não encontradas.")

In [ ]:
# CÉLULA 5: Visualização e Exportação
# Gera um gráfico de dispersão (scatter plot) para visualizar a relação entre as variáveis.

if 'Saldos' in df_merged.columns: # Substitua 'Saldos' se o nome for diferente
    plt.figure(figsize=(10, 6))
    plt.scatter(df_merged['Chuva_Total_Mes'], df_merged['Saldos'], alpha=0.6, color='blue')
    plt.title('Relação entre Precipitação e Saldo de Empregos (RS)')
    plt.xlabel('Precipitação Total Mensal (mm)')
    plt.ylabel('Saldo de Empregos')
    plt.grid(True, linestyle='--', alpha=0.7)
    
    # Salva o gráfico na sua pasta de outputs
    caminho_grafico = os.path.join("..", "outputs", "charts", "correlacao_chuva_emprego.png")
    plt.savefig(caminho_grafico)
    print(f"📊 Gráfico salvo em: {os.path.abspath(caminho_grafico)}")
    plt.show()